# Reflection Pattern Hands-On Code Example

This example implements a reflection loop using the Langchain library and Google GenAI model to iteratively generate and refine a Python function that calculates the factorial of a number. The process starts with a task prompt, generates initial code, and then repeatedly reflects on the code based on critiques from a simulated senior software engineer role, refining the code in each iteration until the critique stage determines the code is perfect or a maximum number of iterations is reached. Finally, it prints the resulting refined code.

In [1]:
# !pip install langchain langchain-community langchain-google-genai langgraph

> Note: Create a `.env` file in the same directory with your Google Generative AI API key:
> ```
> GOOGLE_API_KEY="<your_google_api_key_here>"
> ```

In [2]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage

In [3]:
load_dotenv(override=True)

True

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [5]:
def run_reflection_loop():
    """Demonstrates a multi-step AI reflection loop to progressively improve a Python function."""
    task_prompt = """Your task is to create a Python function named `calculate_factorial`.
    This function should do the following:
    1. Accept a single integer `n` as input.
    2. Calculate its factorial (n!).
    3. Include a clear docstring explaining what the function does.
    4. Handle edge cases: The factorial of 0 is 1.
    5. Handle invalid input: Raise a ValueError if the input is a negative number."""

    # Reflection loop
    max_iterations = 3
    current_code = ""
    message_history = [HumanMessage(content=task_prompt)]

    for i in range(max_iterations):
        print(f"\n{'='*25} REFLECTION LOOP: ITERATION {i+1} {'='*25}\n")

        # 1. Generate/Refine Stage
        if i == 0:
            print("\n>>> STAGE 1: GENERATING initial code...")
            response = llm.invoke(message_history)
            current_code = response.content
        else:
            print("\n>>> STAGE 1: REFINING code based on previous critique...")
            message_history.append(HumanMessage(content="Please refine the code using the critiques provided."))
            response = llm.invoke(message_history)
            current_code = response.content
        
        print(f"\n--- Generated code (v{i+1}) ---\n{current_code}")
        message_history.append(response)

        # 2. REFLECT STAGE
        print("\n>>> STAGE 2: REFLECTING on the generated code...")
        reflection_prompt = [
            SystemMessage(content="""You are a senior software engineer and an expert in Python.
                          Your role is to perform a meticulous code review. Critically evaluate the provided Python code based on the original task requirements.
                          Look for bugs, style issues, missing edge cases, and areas for improvement.
                          If the code is perfect and meets all requirements, respond with the single phrase 'CODE_IS_PERFECT'.
                          Otherwise, provide a bulleted list of your critiques."""),
            HumanMessage(content=f"Original Task:\n{task_prompt}\n\nCode to Review:\n{current_code}")
        ]

        critique_response = llm.invoke(reflection_prompt)
        critique = critique_response.content

        # 3. STOPPING CONDITION
        if "CODE_IS_PERFECT" in critique:
            print("\n--- Critique ---\nNo further critiques found. The code is satisfactory.")
            break
        
        print(f"\n--- Critique ---\n{critique}")
        message_history.append(HumanMessage(content=f"Critique of the previous code:\n{critique}."))
    
    print(f"\n{'='*30} FINAL RESULT {'='*30}")
    print(f"\nFinal refined code after the reflection process:\n{current_code}")

In [6]:
run_reflection_loop()


========================= REFLECTION LOOP: ITERATION 1 =========================


>>> STAGE 1: GENERATING initial code...

--- Generated code (v1) ---
```python
def calculate_factorial(n: int) -> int:
    """
    Calculates the factorial of a non-negative integer.

    The factorial of a non-negative integer n, denoted by n!, is the product
    of all positive integers less than or equal to n.
    The factorial of 0 is defined as 1.

    Args:
        n (int): The non-negative integer for which to calculate the factorial.

    Returns:
        int: The factorial of n.

    Raises:
        ValueError: If the input number `n` is negative.
    """
    # Handle invalid input: negative numbers
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")

    # Handle edge case: factorial of 0
    if n == 0:
        return 1

    # Calculate factorial for positive numbers iteratively
    factorial_result = 1
    for i in range(1, n + 1):
        factorial_result

Before concluding, it's important to consider that while the Reflection pattern significantly enhances output quality, it comes with important trade-offs. The iterative process, though powerful, can lead to higher costs and latency, since every refinement loop may require a new LLM call, making it suboptimal for time-sensitive applications. Furthermore, the pattern is memory-intensive; with each iteration, the conversational history expands, including the initial output, critique, and subsequent refinements.